# 实践实验室：决策树
在本练习中，你将从零开始实现一个决策树，并将其应用于判断蘑菇是可食用还是有毒的分类任务。

# 大纲 
- [1 - 包](#1)
- [2 - 问题陈述](#2)
- [3 - 数据集](#3)
    - [3.1 独热编码数据集](#3.1)
- [4 - 决策树复习](#4)
    - [4.1 计算熵](#4.1)
        - [练习1](#ex01)
    - [4.2 拆分数据集](#4.2)
        - [练习2](#ex02)
    - [4.3 计算信息增益](#4.3)
        - [练习3](#ex03)
    - [4.4 获取最佳拆分](#4.4)
        - [练习4](#ex04)
- [5 - 构建树](#5)


<a name="1"></a>
## 1 - 包
首先，让我们运行下面的代码块，导入本次作业所需的所有包。
- [numpy](www.numpy.org)是Python中处理矩阵的基础包。
- [matplotlib](http://matplotlib.org)是Python中用于绘制图表的著名库。
- ``utils.py``包含本次作业的辅助函数。你无需修改该文件中的代码。


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from public_tests import *

%matplotlib widget

<a name="2"></a>
## 2 - 问题陈述
假设你要创办一家种植和销售野生蘑菇的公司。
- 由于并非所有蘑菇都可食用，你希望能够根据蘑菇的物理属性判断其是否可食用。
- 你拥有一些现有数据可用于此任务。

你能否利用这些数据来帮助确定哪些蘑菇可以安全销售？   

注意：所使用的数据集仅用于说明目的。它并非用于识别可食用蘑菇的指南。


<a name="3"></a>
## 3 - Dataset

你将从加载此任务的数据集开始。你收集的数据集如下：

| Cap Color菌盖颜色 | Stalk Shape菌柄形状 | Solitary单生 | Edible可食用的 |
|:---------:|:-----------:|:--------:|:------:|
|   Brown   |   Tapering  |    Yes   |    1   |
|   Brown   |  Enlarging  |    Yes   |    1   |
|   Brown   |  Enlarging  |    No    |    0   |
|   Brown   |  Enlarging  |    No    |    0   |
|   Brown   |   Tapering  |    Yes   |    1   |
|    Red    |   Tapering  |    Yes   |    0   |
|    Red    |  Enlarging  |    No    |    0   |
|   Brown   |  Enlarging  |    Yes   |    1   |
|    Red    |   Tapering  |    No    |    1   |
|   Brown   |  Enlarging  |    No    |    0   |


- 你有10个蘑菇样本。对于每个样本，你有
    - 三个特征
        - 菌盖颜色（`棕色`或`红色`），
        - 菌柄形状（`渐细`或`膨大`），以及
        - 是否单生（`是`或`否`）
    - 标签
        - 是否可食用（`1`表示可食用，`0`表示有毒）

<a name="3.1"></a>
### 3.1 独热编码数据集
为便于实现，我们对特征进行了独热编码（将其转换为取值为0或1的特征）

| Brown Cap | Tapering Stalk Shape | Solitary | Edible |
|:---------:|:--------------------:|:--------:|:------:|
|     1     |           1          |     1    |    1   |
|     1     |           0          |     1    |    1   |
|     1     |           0          |     0    |    0   |
|     1     |           0          |     0    |    0   |
|     1     |           1          |     1    |    1   |
|     0     |           1          |     1    |    0   |
|     0     |           0          |     0    |    0   |
|     1     |           0          |     1    |    1   |
|     0     |           1          |     0    |    1   |
|     1     |           0          |     0    |    0   |

因此，
- `X_train` 为每个样本包含三个特征
    - 棕色（值为 `1` 表示菌盖颜色为“棕色”，值为 `0` 表示菌盖颜色为“红色”）
    - 渐细形状（值为 `1` 表示菌柄形状为“渐细”，值为 `0` 表示菌柄形状为“膨大”）
    - 单生（值为 `1` 表示“是”，值为 `0` 表示“否”）
- `y_train` 表示蘑菇是否可食用
    - `y = 1` 表示可食用
    - `y = 0` 表示有毒

In [2]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

#### 查看变量
让我们更加熟悉你的数据集。 
- 一个好的开始方式是打印出每个变量，看看它包含什么内容。

下面的代码会打印出 `X_train` 的前几个元素以及该变量的类型。

In [3]:
print("First few elements of X_train:\n", X_train[:5])
print("Type of X_train:",type(X_train))

First few elements of X_train:
 [[1 1 1]
 [1 0 1]
 [1 0 0]
 [1 0 0]
 [1 1 1]]
Type of X_train: <class 'numpy.ndarray'>


现在，让我们对`y_train`做同样的操作。

In [4]:
print("First few elements of y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

First few elements of y_train: [1 1 0 0 1]
Type of y_train: <class 'numpy.ndarray'>


#### 检查变量的维度
另一种熟悉数据的有效方法是查看其维度。   

请打印出`X_train`和`y_train`的形状，看看数据集中有多少个训练样本。

In [5]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(X_train))

The shape of X_train is: (10, 3)
The shape of y_train is:  (10,)
Number of training examples (m): 10


<a name="4"></a>

## 4 - 决策树复习
在本实践实验中，你将根据提供的数据集构建一棵决策树。   
- 回顾一下，构建决策树的步骤如下：
    - 从根节点的所有示例开始
    - 计算对所有可能特征进行分割的信息增益，并选择信息增益最高的特征
    - 根据选定的特征划分数据集，并创建树的左右分支
    - 不断重复划分过程，直到满足停止条件

   
   
- 在本实验中，你将实现以下函数，这些函数将允许你使用信息增益最高的特征将一个节点划分为左右分支
    - 计算节点的熵
    - 根据给定特征在节点处将数据集划分为左右分支
    - 计算基于给定特征划分的信息增益
    - 选择使信息增益最大化的特征

   
- 然后，我们将使用你实现的辅助函数，通过重复划分过程构建决策树，直到满足停止条件
    - 对于本实验，我们选择的停止条件是将最大深度设置为2

<a name="4.1"></a>
### 4.1 计算熵
首先，你要编写一个名为`compute_entropy`的辅助函数，用于计算节点处的熵（杂质度量）。 
- 该函数接受一个NumPy数组（`y`），该数组指示该节点中的示例是可食用的（`1`）还是有毒的（`0`）。

完成下面的`compute_entropy()`函数，以：
* 计算$p_1$，即可食用示例的比例（即在`y`中值为`1`的示例）
* 然后，熵的计算公式为：

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$
* 注意
    * 对数以$2$为底进行计算
    * 出于实现目的，$0\text{log}_2(0) = 0$。也就是说，如果`p_1 = 0`或`p_1 = 1`，则将熵设置为`0`
    * 确保检查节点处的数据不为空（即`len(y)!= 0`）。
如果为空，则返回`0`
    
<a name="ex01"></a>
### 练习1   

请按照前面的说明完成`compute_entropy()`函数。   

如果你遇到困难，可以查看下面单元格之后给出的提示，以帮助你完成实现。

In [6]:
# UNQ_C1
# GRADED FUNCTION: compute_entropy

def compute_entropy(y):
    """
    Computes the entropy for 
    
    Args:
       y (ndarray): Numpy array indicating whether each example at a node is
           edible (`1`) or poisonous (`0`)
       
    Returns:
        entropy (float): Entropy at that node
        
    """
    # You need to return the following variables correctly
    entropy = 0.
    
    ### START CODE HERE ###
    if len(y) != 0:
        p1 = len(y[y == 1]) / len(y)

        if p1 != 0 and p1 != 1:
            entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)
        else:
            entropy = 0.
           
    ### END CODE HERE ###        
    
    return entropy

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
    
   * To calculate `p1`
       * You can get the subset of examples in `y` that have the value `1` as `y[y == 1]`
       * You can use `len(y)` to get the number of examples in `y`
   * To calculate `entropy`
       * <a href="https://numpy.org/doc/stable/reference/generated/numpy.log2.html">np.log2</a> let's you calculate the logarithm to base 2 for a numpy array
       * If the value of `p1` is 0 or 1, make sure to set the entropy to `0` 
     
<details>
          <summary><font size="2" color="darkblue"><b> Click for more hints</b></font></summary>
        
    * Here's how you can structure the overall implementation for this function   
    
```python 
    def compute_entropy(y):
        
        # You need to return the following variables correctly
        entropy = 0.

        ### START CODE HERE ###
        if len(y) != 0:
            # Your code here to calculate the fraction of edible examples (i.e with value = 1 in y)
            p1 =

            # For p1 = 0 and 1, set the entropy to 0 (to handle 0log0)
            if p1 != 0 and p1 != 1:
                # Your code here to calculate the entropy using the formula provided above
                entropy = 
            else:
                entropy = 0. 
        ### END CODE HERE ###        

        return entropy
```

    If you're still stuck, you can check the hints presented below to figure out how to calculate `p1` and `entropy`.
    
<details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate p1</b></font></summary>
           &emsp; &emsp; You can compute p1 as <code>p1 = len(y[y == 1]) / len(y) </code>
</details>

<details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate entropy</b></font></summary>
          &emsp; &emsp; You can compute entropy as <code>entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)</code>
    </details>
        
    </details>

</details>

    


你可以通过运行以下测试代码来检查你的实现是否正确：

In [7]:
# Compute entropy at the root node (i.e. with all examples)
# Since we have 5 edible and 5 non-edible mushrooms, the entropy should be 1"

print("Entropy at root node: ", compute_entropy(y_train)) 

# UNIT TESTS
compute_entropy_test(compute_entropy)

Entropy at root node:  1.0
 All tests passed.


**Expected Output**:
<table>
  <tr>
    <td> <b>Entropy at root node:<b> 1.0 </td> 
  </tr>
</table>

<a name="4.2"></a>
### 4.2 拆分数据集

接下来，你将编写一个名为 `split_dataset` 的辅助函数，该函数接受一个节点处的数据以及要依据其进行拆分的特征，并将数据拆分为左分支和右分支。在本实验稍后部分，你将编写代码来计算拆分的优劣程度。   

- 该函数接受训练数据、该节点处数据点的索引列表以及要依据其进行拆分的特征。
- 它对数据进行拆分，并返回左分支和右分支的索引子集。
- 例如，假设我们从根节点开始（因此 `node_indices = [0,1,2,3,4,5,6,7,8,9]`），并且我们选择依据特征 `0` 进行拆分，该特征表示样本是否有棕色菌盖。
    - 那么该函数的输出就是，`left_indices = [0,1,2,3,4,7,9]` 以及 `right_indices = [5,6,8]` 
    
| Index | Brown Cap | Tapering Stalk Shape | Solitary | Edible |
|:-----:|:---------:|:--------------------:|:--------:|:------:|
|   0   |     1     |           1          |     1    |    1   |
|   1   |     1     |           0          |     1    |    1   |
|   2   |     1     |           0          |     0    |    0   |
|   3   |     1     |           0          |     0    |    0   |
|   4   |     1     |           1          |     1    |    1   |
|   5   |     0     |           1          |     1    |    0   |
|   6   |     0     |           0          |     0    |    0   |
|   7   |     1     |           0          |     1    |    1   |
|   8   |     0     |           1          |     0    |    1   |
|   9   |     1     |           0          |     0    |    0   |

<a name="ex02"></a>
### 练习2

请完成如下所示的`split_dataset()`函数。   

- 对于`node_indices`中的每个索引
    - 如果`X`中该索引处对应特征的值为`1`，则将该索引添加到`left_indices`中。
    - 如果`X`中该索引处对应特征的值为`0`，则将该索引添加到`right_indices`中。

如果你遇到困难，可以查看下面单元格之后给出的提示，以帮助你完成实现。

In [8]:
# UNQ_C2
# GRADED FUNCTION: split_dataset

def split_dataset(X, node_indices, feature):
    """
    Splits the data at the given node into
    left and right branches
    
    Args:
        X (ndarray):             Data matrix of shape(n_samples, n_features)
        node_indices (ndarray):  List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        left_indices (ndarray): Indices with feature value == 1
        right_indices (ndarray): Indices with feature value == 0
    """
    
    # You need to return the following variables correctly
    left_indices = []
    right_indices = []
    
    ### START CODE HERE ###
    for i in node_indices:   
        if X[i][feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
           
    ### END CODE HERE ###
        
    return left_indices, right_indices

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
    
   * Here's how you can structure the overall implementation for this function

```python 
    def split_dataset(X, node_indices, feature):
    
        # You need to return the following variables correctly
        left_indices = []
        right_indices = []

        ### START CODE HERE ###
        # Go through the indices of examples at that node
        for i in node_indices:   
            if # Your code here to check if the value of X at that index for the feature is 1
                left_indices.append(i)
            else:
                right_indices.append(i)
        ### END CODE HERE ###
        
    return left_indices, right_indices

```
<details>
<summary><font size="2" color="darkblue"><b> Click for more hints</b></font></summary>
        
    The condition is <code> if X[i][feature] == 1:</code>.
        
</details>

</details>

    


现在，让我们使用下面的代码块检查你的实现。让我们尝试在根节点拆分数据集，正如我们上面所讨论的，根节点包含特征0（棕色菌盖）的所有示例。

In [9]:
root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Feel free to play around with these variables
# The dataset only has three features, so this value can be 0 (Brown Cap), 1 (Tapering Stalk Shape) or 2 (Solitary)
feature = 0

left_indices, right_indices = split_dataset(X_train, root_indices, feature)

print("Left indices: ", left_indices)
print("Right indices: ", right_indices)

# UNIT TESTS    
split_dataset_test(split_dataset)

Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
 All tests passed.


**Expected Output**:
```
Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
```

<a name="4.3"></a>
### 4.3 计算信息增益

接下来，你将编写一个名为 `information_gain` 的函数，该函数接受训练数据、节点处的索引以及要进行分割的特征，并返回分割后的信息增益。

<a name="ex03"></a>
### 练习3
请完成下面所示的`compute_information_gain()`函数，以计算

$$\text{Information Gain} = H(p_1^\text{node})- (w^{\text{left}}H(p_1^\text{left}) + w^{\text{right}}H(p_1^\text{right}))$$

其中 
- $H(p_1^\text{node})$ 节点处的熵 
- $H(p_1^\text{left})$ and $H(p_1^\text{right})$ 分别是分割后左右分支的熵
- $w^{\text{left}}$ and $w^{\text{right}}$ 分别是左右分支上示例的比例

注意：
- 你可以使用上面实现的`compute_entropy()`函数来计算熵。
- 我们提供了一些起始代码，这些代码使用你上面实现的`split_dataset()`函数来分割数据集。

如果你遇到困难，可以查看下面单元格之后给出的提示，以帮助你完成实现。

In [10]:
# UNQ_C3
# GRADED FUNCTION: compute_information_gain

def compute_information_gain(X, y, node_indices, feature):
    
    """
    Compute the information of splitting the node on a given feature
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
   
    Returns:
        cost (float):        Cost computed
    
    """    
    # Split dataset
    left_indices, right_indices = split_dataset(X, node_indices, feature)
    
    # Some useful variables
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    # You need to return the following variables correctly
    information_gain = 0
    
    ### START CODE HERE ###
    node_entropy = compute_entropy(y_node)
    left_entropy = compute_entropy(y_left)
    right_entropy = compute_entropy(y_right)

    w_left = len(X_left) / len(X_node)
    
    w_right = len(X_right) / len(X_node)

    weighted_entropy = w_left * left_entropy + w_right * right_entropy

    information_gain =  node_entropy - weighted_entropy                                               
    
    ### END CODE HERE ###  
    
    return information_gain

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
    
   * Here's how you can structure the overall implementation for this function

```python 
    def compute_information_gain(X, y, node_indices, feature):
        # Split dataset
        left_indices, right_indices = split_dataset(X, node_indices, feature)

        # Some useful variables
        X_node, y_node = X[node_indices], y[node_indices]
        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        # You need to return the following variables correctly
        information_gain = 0

        ### START CODE HERE ###
        # Your code here to compute the entropy at the node using compute_entropy()
        node_entropy = 
        # Your code here to compute the entropy at the left branch
        left_entropy = 
        # Your code here to compute the entropy at the right branch
        right_entropy = 

        # Your code here to compute the proportion of examples at the left branch
        w_left = 
        
        # Your code here to compute the proportion of examples at the right branch
        w_right = 

        # Your code here to compute weighted entropy from the split using 
        # w_left, w_right, left_entropy and right_entropy
        weighted_entropy = 

        # Your code here to compute the information gain as the entropy at the node
        # minus the weighted entropy
        information_gain = 
        ### END CODE HERE ###  

        return information_gain

```
    If you're still stuck, check out the hints below.
    
<details>
        <summary><font size="2" color="darkblue"><b> Hint to calculate the entropies</b></font></summary>
        
<code>node_entropy = compute_entropy(y_node)</code><br>
<code>left_entropy = compute_entropy(y_left)</code><br>
<code>right_entropy = compute_entropy(y_right)</code>
        
</details>
    
<details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate w_left and w_right</b></font></summary>
           <code>w_left = len(X_left) / len(X_node)</code><br>
           <code>w_right = len(X_right) / len(X_node)</code>
    </details>
    
<details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate weighted_entropy</b></font></summary>
           <code>weighted_entropy = w_left * left_entropy + w_right * right_entropy</code>
    </details>
    
<details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate information_gain</b></font></summary>
           <code> information_gain = node_entropy - weighted_entropy</code>
    </details>


</details>


你现在可以使用下面的单元格检查你的实现，并计算基于每个特征进行划分所得到的信息增益是多少。

In [11]:
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print("Information Gain from splitting the root on brown cap: ", info_gain0)
    
info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)

# UNIT TESTS
compute_information_gain_test(compute_information_gain)

Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
 All tests passed.


**Expected Output**:
```
Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
```

在根节点处按“独居”（特征2）进行划分可获得最大信息增益。因此，它是根节点处进行划分的最佳特征。

<a name="4.4"></a>
### 4.4 获取最佳分割
现在，让我们编写一个函数，通过像上面那样计算每个特征的信息增益，来获取用于分割的最佳特征，并返回能提供最大信息增益的特征。 

<a name="ex04"></a>
### 练习4 
请完成下面所示的`get_best_split()`函数。 
- 该函数接收训练数据以及该节点处数据点的索引
- 函数的输出是能提供最大信息增益的特征
    - 你可以使用`compute_information_gain()`函数遍历各个特征，并计算每个特征的信息

如果你遇到困难，可以查看下面单元格之后给出的提示，以帮助你完成实现。 

In [12]:
# UNQ_C4
# GRADED FUNCTION: get_best_split

def get_best_split(X, y, node_indices):   
    """
    Returns the optimal feature and threshold value
    to split the node data 
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.

    Returns:
        best_feature (int):     The index of the best feature to split
    """    
    
    # Some useful variables
    num_features = X.shape[1]
    
    # You need to return the following variables correctly
    best_feature = -1
    
    ### START CODE HERE ###
    max_info_gain = 0
    
    for feature in range(num_features): 
        info_gain = compute_information_gain(X, y, node_indices, feature)
        if info_gain > max_info_gain:  
            max_info_gain = info_gain
            best_feature = feature
       
    ### END CODE HERE ##    
   
    return best_feature

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
    
   * Here's how you can structure the overall implementation for this function
    
```python 
    def get_best_split(X, y, node_indices):   

        # Some useful variables
        num_features = X.shape[1]

        # You need to return the following variables correctly
        best_feature = -1

        ### START CODE HERE ###
        max_info_gain = 0

        # Iterate through all features
        for feature in range(num_features): 
            
            # Your code here to compute the information gain from splitting on this feature
            info_gain = 
            
            # If the information gain is larger than the max seen so far
            if info_gain > max_info_gain:  
                # Your code here to set the max_info_gain and best_feature
                max_info_gain = 
                best_feature = 
        ### END CODE HERE ##    
   
    return best_feature
```

If you're still stuck, check out the hints below.
    
<details>
    <summary><font size="2" color="darkblue"><b> Hint to calculate info_gain</b></font></summary>
        
<code>info_gain = compute_information_gain(X, y, node_indices, feature)</code>
</details>
    
<details>
          <summary><font size="2" color="darkblue"><b>Hint to update the max_info_gain and best_feature</b></font></summary>
           <code>max_info_gain = info_gain</code><br>
           <code>best_feature = feature</code>
    </details>
</details>


现在，让我们使用下面的单元格检查你函数的实现情况。

In [13]:
best_feature = get_best_split(X_train, y_train, root_indices)
print("Best feature to split on: %d" % best_feature)

# UNIT TESTS
get_best_split_test(get_best_split)

Best feature to split on: 2
 All tests passed.


正如我们上面所见，该函数返回在根节点进行分割的最佳特征是特征2（“独居”）。

<a name="5"></a>
## 5 - 构建决策树   

在本节中，我们将使用你上面实现的函数，通过依次选择最佳特征进行划分，直至达到停止条件（最大深度为2），从而生成一棵决策树。   

此部分你无需实现任何内容。

In [14]:
# Not graded
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    """
    Build a tree using the recursive algorithm that split the dataset into 2 subgroups at each node.
    This function just prints the tree.
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        branch_name (string):   Name of the branch. ['Root', 'Left', 'Right']
        max_depth (int):        Max depth of the resulting tree. 
        current_depth (int):    Current depth. Parameter used during recursive call.
   
    """ 

    # Maximum depth reached - stop splitting
    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   
    # Otherwise, get best split and split the data
    # Get the best feature and threshold at this node
    best_feature = get_best_split(X, y, node_indices) 
    tree.append((current_depth, branch_name, best_feature, node_indices))
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # Split the dataset at the best feature
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    
    # continue splitting the left and the right child. Increment current depth
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)


In [15]:
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)

 Depth 0, Root: Split on feature: 2
- Depth 1, Left: Split on feature: 0
  -- Left leaf node with indices [0, 1, 4, 7]
  -- Right leaf node with indices [5]
- Depth 1, Right: Split on feature: 1
  -- Left leaf node with indices [8]
  -- Right leaf node with indices [2, 3, 6, 9]
